In [48]:
import vocab_creator
import pandas as pd
import torch
import Adversary_Passes as Adv
import vocab_creator
import bangla_corpus_builder as bcb

In [40]:
dataset=pd.read_csv(r"C:\project\DL project\Dataset_Creator\bangla_corpus\sentences.csv")
# print(dataset)

In [41]:
########## Required Inputs ###################
# Vocabulary
vocab=torch.load(r"C:\project\DL project\vocab.pth",map_location="cpu")

# Trained Weights
weights=torch.load(r"C:\project\DL project\weights.pth",map_location="cpu")

# Author_vocab
author_vocab={"Bibhutibhushan Bandopadhyay":0,"Rabindranath Tagore":1,"Sarat Chandra Chattopadhyay":2,"Satyajit Ray":3,"Sunil Gangopadhay":4}

# Special Token Indexes
pad_id=0
unk_id=1
eos_id=2

print(vocab.keys())

dict_keys(['<pad>', '<unk>', '<eos>', 'জগতের', 'যে', 'পথে', 'সভ্য', 'মানুষের', 'চলাচল', 'কম,', 'কত', 'অদ্ভুত', 'স্রোত', 'আপন', 'মনে', 'অজানা', 'দিয়া', 'করিয়া', 'বহিয়া', 'চলে', 'সে', 'তাহাদের', 'সহিত', 'আজও', 'ভুলিতে', 'পারি', 'নাই৷', 'কিন্তু', 'আমার', 'এ', 'আনন্দের', 'নয়,', 'দুঃখের', 'এই', 'প্রকৃতির', 'হাতেই', 'হইয়াছিল,', 'বনের', 'সেজন্য', 'আমায়', 'কখনো', 'ক্ষমা', 'করিবেন', 'না', 'জানি', 'নিজের', 'অপরাধের', 'কথা', 'মুখে', 'বলিলে', 'ভার', 'হইয়া', 'যায়', 'তাই', 'প্রথম', 'পরিচ্ছেদ', 'বছর', 'আগেকার', 'বি', 'পাশ', 'কলিকাতায়', 'বসিয়া', 'আছি', 'বহু', 'জায়গায়', 'চাকুরি', 'অনেকদিন', 'ধরিয়া', 'নিতান্ত', 'দেয়', 'না,', 'উপর', 'মেসের', 'ম্যানেজার', 'অস্থির', 'পূজা', 'মন্দ', 'সকালে', 'উঠিয়া', 'আজ', 'সব', 'বন্ধ,', 'দু-একটা', 'একটু', 'আশা', 'তা', 'আর', 'কোথাও', 'যাওয়া', 'কোন', 'কাজের', 'হইবে', 'বরং', 'তার', 'চেয়ে', 'ঘুরিয়া', 'ঠাকুর', 'দেখিয়া', 'চাকর', 'এমন', 'সময়', 'কাগজ', 'হাতে', 'গেল', 'পড়িয়া', 'দেখিলাম', 'লেখা', 'চিঠি', 'ভালো', 'ব্যবস্থা', 'হইয়াছে,', 'কাছে', 'টাকা', 'আমি', 'য

In [42]:
# Load trained Weights into model

# In PyTorch, load_state_dict is a method used to load a model's or optimizer's parameter dictionary (the state_dict) into an object. 
# It is the recommended way to restore saved models because it offers the most flexibility for future use

Adv.embed.load_state_dict(weights['embed'])
Adv.gru.load_state_dict(weights['encoder_gru'])
Adv.decoder_gru.load_state_dict(weights['decoder_gru'])
Adv.decoder_linear.load_state_dict(weights["decoder_linear"])
Adv.emb_author.load_state_dict(weights['emb_author'])

# Use eval() to switch on evalution mode from training mode

Adv.embed.eval()
Adv.gru.eval()
Adv.decoder_gru.eval()
Adv.decoder_linear.eval()
Adv.emb_author.eval()

Embedding(5, 128)

# `.eval()` in PyTorch — Explanation Note

## 1. Purpose

`.eval()` is used to switch a model from **training mode → inference mode**.

It tells the model:

> “Stop learning, start behaving deterministically.”

---

## 2. Why this is needed

Some neural network layers behave differently during training and testing.

### During training:

* Model introduces randomness or uses batch-specific statistics
* Helps generalization and learning

### During inference:

* Model must behave **consistently and predictably**

---

## 3. What `.eval()` actually does

Internally, it sets:

```
model.training = False
```

This changes how certain layers operate.

---

## 4. Layers affected by `.eval()`

### (a) Dropout

* Training: randomly drops neurons
* Eval: uses full network (no randomness)

### (b) Batch Normalization

* Training: uses current batch statistics
* Eval: uses learned running averages

---

## 5. Your current model case

In your architecture:

* No Dropout
* No BatchNorm

So `.eval()` does not visibly change output **right now**

BUT:

* Still required as a best practice
* Prevents future bugs if architecture changes

---

## 6. `.eval()` vs `torch.no_grad()`

| Feature                       | `.eval()` | `torch.no_grad()` |
| ----------------------------- | --------- | ----------------- |
| Changes layer behavior        | ✔         | ✘                 |
| Disables gradient computation | ✘         | ✔                 |

### Correct inference usage:

```
model.eval()

with torch.no_grad():
    output = model(input)
```

---

## 7. What happens if you don’t use `.eval()`

* Outputs may become unstable (if dropout present)
* Predictions may vary for same input
* Model behaves like it is still training

---

## 8. Key intuition

Training mode:
→ model is **learning (with noise and adaptation)**

Evaluation mode:
→ model is **fixed and reliable**

---

## 9. One-line summary

`.eval()` ensures the model behaves like a **final trained system**, not a learning system.


In [43]:
################ Generation Part ####################################

In [55]:
# Tokenization of Input
# Input Sentence
input_sentence = input("Enter Your input sentence: ")

sentence=bcb.clean_text(input_sentence)

splits=sentence.split()

tokens=[]


for i in splits:
    if i not in vocab.keys():
        tokens.append(vocab['<unk>'])
    else:
        tokens.append(vocab[i])

if len(tokens)==0:
    print("Sentence has been completely omitted post cleaning and toenization due to noise")
else:
    print(torch.tensor(tokens).unsqueeze(0))     # Converts rank 1 tensor into rank 2

tensor([[1887,    1,    1,  958, 1484]])
